In [1]:
# Get raw data
with open('input/21.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
# Part 1
dirdict = {'^':(-1,0),'v':(1,0),'<':(0,-1),'>':(0,1)}

class Keypad(object):
    types = {'num': '789,456,123, 0A',
             'dir': ' ^A,<v>'}
    def __init__(self, kptype):
        self.keys_locs = {c:(i,j)
                          for i,r in enumerate(self.types[kptype].split(','))
                          for j,c in enumerate(r) 
                          if c != ' '}
        
    def get_paths(self,fromkey,tokey):
        frompos,topos = [self.keys_locs[i] for i in [fromkey,tokey]]
        topush = [k
                  for i,j in enumerate(['^_v','<_>'])
                  for z in [topos[i]-frompos[i]]
                  for k in ([(j[z//abs(z)+1],abs(z))] if z else [])]
        return self.get_paths_inner(frompos,topush)
    
    def get_paths_inner(self,pos,topush):
        if topush:
            return [topush[i][0]*topush[i][1]+j
                    for i in range(len(topush))
                    for npos in [tuple([k+topush[i][1]*l
                                        for k,l in zip(pos,
                                                       dirdict[topush[i][0]])])]
                    if npos in self.keys_locs.values()
                    for j in self.get_paths_inner(npos, topush[:i]+topush[i+1:])]
        else:
            return ['A']
        
    def get_sequence_paths(self,sequence):
        return [self.get_paths(z if i else 'A', (z:=j)) 
                for i,j in enumerate(sequence)]
        
def get_all_combos(paths):
    if paths:
        return [i+j
                for i in paths[0]
                for j in get_all_combos(paths[1:])]
    else:
        return ['']
    
numeric = Keypad('num')
directional = Keypad('dir')

codes = [i for i in rawinput.split('\n')]
sum([int(c.replace('A',''))
     * min([len(k)
            for i in get_all_combos(numeric.get_sequence_paths(c))
            for j in get_all_combos(directional.get_sequence_paths(i))
            for k in get_all_combos(directional.get_sequence_paths(j))])
     for c in codes])

217662

In [3]:
# Part 2
def min_seq_len(subseq, layers):
    dkey = (subseq, layers)
    if dkey not in known:
        if layers:
            known[dkey] = min([sum([min_seq_len(f'{s}A', layers-1)
                                    for s in p.strip('A').split('A')])
                               for p in get_all_combos(directional.get_sequence_paths(subseq))])
        else:
            known[dkey] = len(subseq)
    return known[dkey]

known = dict()
sum([int(c.replace('A',''))
     * min([sum([min_seq_len(f'{s}A', 25) 
                 for s in p.strip('A').split('A')])
            for p in get_all_combos(numeric.get_sequence_paths(c))])
     for c in codes])

263617786809000